<a href="https://colab.research.google.com/github/MarkowitzMx/Programacion-para-analitica-descriptiva-y-predictiva-2026/blob/main/Copy_of_S06_Actividad_01_Calificada_NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 06 — Actividad calificable: NumPy y preparación de datos

# Actividad calificada — NumPy y preparación de datos

**Programación para Analítica Descriptiva y Predictiva**

Maestría en Inteligencia Artificial y Analítica de Datos — UACJ

---


**Nombre completo:**  Cristobal Lemus
**Matrícula:**  274690

---

**Entrega:** individual, en este notebook de Google Colab (comparte el enlace con permiso de edición para el docente).
**Plazo:** 7 días naturales a partir de la publicación de esta actividad.
**Penalización por entrega tardía:** 5% por día, hasta 7 días; después de eso, la actividad no se recibe.
**Valor total:** 100 puntos.

### Herramientas permitidas

Esta actividad evalúa lo visto hasta la Sesión 6: expresiones regulares (`re`), comprensión de listas, `map()`, `lambda`, y NumPy. **No uses pandas ni scikit-learn** — todavía no se han visto en el curso, y su uso no sumará puntos aunque el resultado sea correcto.

### Cómo se evalúa

Cada ejercicio indica su valor en puntos. Se evalúa el resultado correcto **y** el uso apropiado de las herramientas indicadas en cada enunciado — una solución correcta que ignore la herramienta pedida (por ejemplo, usar un bucle `for` donde se pide `map()` o una operación vectorizada) no obtiene el puntaje completo. Comenta tu código donde el paso no sea evidente.

## Librerías

Ejecuta esta celda antes de empezar.

In [12]:
import re
import numpy as np

## Datos de partida

Una red de 4 estaciones meteorológicas registró temperatura y humedad durante 5 días. Los datos llegaron en un formato de texto crudo, como ocurre frecuentemente al recibir información de sensores o sistemas externos. Ejecuta la siguiente celda para cargar las lecturas — **no la modifiques**.

In [13]:
lecturas_crudas = [
    "EST-01 | Dia1 | Temp: 22.3C | Hum: 60%",
    "EST-01 | Dia2 | Temp: 19.8C | Hum: 64%",
    "EST-01 | Dia3 | Temp: 24.1C | Hum: 58%",
    "EST-01 | Dia4 | Temp: 26.5C | Hum: 55%",
    "EST-01 | Dia5 | Temp: 21.0C | Hum: 63%",
    "EST-02 | Dia1 | Temp: 24.0C | Hum: 70%",
    "EST-02 | Dia2 | Temp: 23.5C | Hum: 72%",
    "EST-02 | Dia3 | Temp: 27.2C | Hum: 65%",
    "EST-02 | Dia4 | Temp: 20.1C | Hum: 74%",
    "EST-02 | Dia5 | Temp: 22.8C | Hum: 69%",
    "EST-03 | Dia1 | Temp: 18.5C | Hum: 80%",
    "EST-03 | Dia2 | Temp: 19.2C | Hum: 78%",
    "EST-03 | Dia3 | Temp: 20.0C | Hum: 77%",
    "EST-03 | Dia4 | Temp: 21.5C | Hum: 75%",
    "EST-03 | Dia5 | Temp: 19.9C | Hum: 79%",
    "EST-04 | Dia1 | Temp: 27.8C | Hum: 50%",
    "EST-04 | Dia2 | Temp: 26.3C | Hum: 52%",
    "EST-04 | Dia3 | Temp: 25.9C | Hum: 53%",
    "EST-04 | Dia4 | Temp: 24.0C | Hum: 56%",
    "EST-04 | Dia5 | Temp: 28.1C | Hum: 49%",
]

print(len(lecturas_crudas), "lecturas cargadas")
print(lecturas_crudas[0])

20 lecturas cargadas
EST-01 | Dia1 | Temp: 22.3C | Hum: 60%


---

## Ejercicio 1 — Extracción con expresiones regulares (10 pts)

Usando `re` y comprensión de listas, extrae de cada cadena en `lecturas_crudas` el identificador de estación (por ejemplo `'EST-01'`) y el valor de temperatura como texto (por ejemplo `'22.3'`, sin la `C`). El resultado debe ser una lista de tuplas `(estacion, temperatura_texto)`, en el mismo orden que `lecturas_crudas`.

*Por qué importa: en la práctica, los datos casi nunca llegan ya estructurados — extraerlos de texto libre es el primer paso de cualquier pipeline de análisis.*

In [14]:
# Tu código aquí
# extraidos = [...]
# Patrón de expresión regular:
# EST-\d+       -> busca "EST-" seguido de uno o más dígitos (el identificador de estación)
# .*?           -> cualquier caracter, cualquier cantidad de veces
# Temp:\s*      -> la palabra "Temp:" seguida de cero o más espacios
# ([\d.]+)      -> uno o más dígitos o puntos decimales (esto captura la temperatura)
patron = r"(EST-\d+).*?Temp:\s*([\d.]+)C"

# Comprensión de listas: recorremos cada cadena en lecturas_crudas
# re.search(patron, cadena) busca la primera coincidencia del patrón en la cadena
# .groups() devuelve una tupla con los grupos capturados por los paréntesis: (estacion, temperatura_texto)
extraidos = [re.search(patron, linea).groups() for linea in lecturas_crudas]

# Mostramos el resultado para verificar
print(extraidos)

[('EST-01', '22.3'), ('EST-01', '19.8'), ('EST-01', '24.1'), ('EST-01', '26.5'), ('EST-01', '21.0'), ('EST-02', '24.0'), ('EST-02', '23.5'), ('EST-02', '27.2'), ('EST-02', '20.1'), ('EST-02', '22.8'), ('EST-03', '18.5'), ('EST-03', '19.2'), ('EST-03', '20.0'), ('EST-03', '21.5'), ('EST-03', '19.9'), ('EST-04', '27.8'), ('EST-04', '26.3'), ('EST-04', '25.9'), ('EST-04', '24.0'), ('EST-04', '28.1')]


---

## Ejercicio 2 — Construcción del arreglo numérico (10 pts)

A partir de las temperaturas extraídas en el Ejercicio 1 (que están en texto), usa `map()` junto con una conversión a `float` para construir un `ndarray` de tipo numérico. Verifica e imprime su `dtype`.

*Por qué importa: conectar la extracción de texto con un tipo de dato numérico utilizable es un paso que se repite constantemente al preparar datos.*

In [15]:
# Tu código aquí
# temperaturas = np.array(...)
# extraidos es la lista de tuplas (estacion, temperatura_texto) del Ejercicio 1.
# Usamos una comprensión de listas para quedarnos solo con el segundo elemento
# de cada tupla (índice [1]), que es el texto de la temperatura, por ejemplo '22.3'.
temperaturas_texto = [t[1] for t in extraidos]

# map(float, temperaturas_texto) aplica la función float() a cada elemento
# de temperaturas_texto, uno por uno, convirtiendo cada texto ('22.3') a un
# número decimal (22.3). map() devuelve un objeto "map" (un iterador perezoso,
# no una lista todavía), así que hay que convertirlo explícitamente.
temperaturas = np.array(list(map(float, temperaturas_texto)))

# .dtype es un atributo de los arrays de NumPy que indica el tipo de dato
# almacenado internamente (por ejemplo float64, int32, etc.)
print("dtype:", temperaturas.dtype)

# Mostramos el arreglo completo para verificar visualmente el resultado
print(temperaturas)

dtype: float64
[22.3 19.8 24.1 26.5 21.  24.  23.5 27.2 20.1 22.8 18.5 19.2 20.  21.5
 19.9 27.8 26.3 25.9 24.  28.1]


---

## Ejercicio 3 — Organización en una matriz 2D (10 pts)

Reorganiza el arreglo de temperaturas del Ejercicio 2 en una matriz de 4 filas (una por estación) × 5 columnas (una por día) usando `reshape()`. Los datos ya están ordenados por estación en `lecturas_crudas`, así que no necesitas reordenarlos manualmente. Reporta `shape`, `ndim`, `size` y `dtype` de la matriz resultante.

In [16]:
# Tu código aquí
# matriz_temp = temperaturas.reshape(...)
# reshape(4, 5) reorganiza el arreglo unidimensional 'temperaturas' (que tiene 20 elementos)
# en una matriz de 4 filas x 5 columnas, sin cambiar los datos ni su orden,
# solo la "forma" en que se acomodan en memoria.
# Como lecturas_crudas ya viene ordenado por estación (5 lecturas seguidas de EST-01,
# luego 5 de EST-02, etc.), cada fila del resultado corresponde exactamente a una estación,
# y cada columna corresponde a un día (Dia1 a Dia5).
matriz_temp = temperaturas.reshape(4, 5)

# .shape es un atributo que devuelve una tupla con el número de elementos en cada
# dimensión del arreglo. Para una matriz de 4 filas x 5 columnas, será (4, 5).
print("shape:", matriz_temp.shape)

# .ndim es un atributo que indica el número de dimensiones (ejes) del arreglo.
# Un arreglo 1D (como 'temperaturas') tiene ndim=1; una matriz 2D tiene ndim=2.
print("ndim:", matriz_temp.ndim)

# .size es un atributo que indica el número total de elementos en el arreglo,
# sin importar cómo estén distribuidos en filas y columnas.
# Aquí debe ser 4 * 5 = 20, el mismo número de elementos que tenía 'temperaturas'.
print("size:", matriz_temp.size)

# .dtype indica el tipo de dato de los elementos almacenados.
# reshape() no cambia el tipo de dato, así que debe seguir siendo el mismo
# que tenía 'temperaturas' (float64).
print("dtype:", matriz_temp.dtype)

# Mostramos la matriz completa para verificar visualmente que cada fila
# corresponde a una estación y cada columna a un día.
print(matriz_temp)

shape: (4, 5)
ndim: 2
size: 20
dtype: float64
[[22.3 19.8 24.1 26.5 21. ]
 [24.  23.5 27.2 20.1 22.8]
 [18.5 19.2 20.  21.5 19.9]
 [27.8 26.3 25.9 24.  28.1]]


---

## Ejercicio 4 — Indexado, slicing y transposición (10 pts)

A partir de `matriz_temp`:

1. Extrae la fila completa correspondiente a `EST-03`.
2. Extrae la columna completa correspondiente al Día 4.
3. Extrae la submatriz de las primeras 2 estaciones durante los primeros 3 días.
4. Transpón la matriz con `.T` e imprime su nueva forma (`shape`). En una celda de texto (Markdown) o un comentario, explica en qué situación tendría sentido trabajar con los días como filas y las estaciones como columnas.

In [17]:
# Tu código aquí
# 1. Extraer la fila completa de EST-03
# Recordemos que en matriz_temp, la fila 0 = EST-01, fila 1 = EST-02,
# fila 2 = EST-03, fila 3 = EST-04 (porque así estaban ordenados los datos originales).
# matriz_temp[2] selecciona la fila con índice 2 (la tercera fila), es decir EST-03.
# Esto devuelve un arreglo 1D con las 5 temperaturas de esa estación (una por día).
fila_est03 = matriz_temp[2]
print("EST-03:", fila_est03)

# 2. Extraer la columna completa del Día 4
# Las columnas representan los días: columna 0 = Dia1, columna 1 = Dia2,
# columna 2 = Dia3, columna 3 = Dia4, columna 4 = Dia5.
# matriz_temp[:, 3] usa slicing: el ":" antes de la coma significa
# "todas las filas", y el "3" después de la coma selecciona la columna de índice 3 (Día 4).
columna_dia4 = matriz_temp[:, 3]
print("Día 4:", columna_dia4)

# 3. Extraer la submatriz de las primeras 2 estaciones y los primeros 3 días
# matriz_temp[:2, :3] es slicing en ambos ejes a la vez:
#   ":2" en la posición de filas significa "desde el inicio hasta el índice 2 (sin incluirlo)",
#        es decir, filas 0 y 1 -> EST-01 y EST-02.
#   ":3" en la posición de columnas significa "desde el inicio hasta el índice 3 (sin incluirlo)",
#        es decir, columnas 0, 1 y 2 -> Dia1, Dia2, Dia3.
# El resultado es una submatriz de 2 filas x 3 columnas.
submatriz = matriz_temp[:2, :3]
print("Submatriz (2 estaciones, 3 días):\n", submatriz)

# 4. Transponer la matriz
# El atributo .T invierte los ejes de la matriz: lo que antes eran filas
# ahora son columnas y viceversa. No modifica matriz_temp original,
# sino que devuelve una nueva "vista" con los ejes intercambiados.
matriz_transpuesta = matriz_temp.T
print("Shape original:", matriz_temp.shape)
print("Shape transpuesta:", matriz_transpuesta.shape)

# Explicación:
# Trabajar con los días como filas y las estaciones como columnas tendría sentido,
# por ejemplo, si quisiéramos analizar la evolución temporal día a día
# (por ejemplo, calcular el promedio de todas las estaciones para cada día,
# o graficar una serie de tiempo), ya que en muchas herramientas y convenciones
# de análisis de datos cada FILA representa una "observación" (un día) y cada
# COLUMNA representa una "variable" (una estación). Esta orientación facilita
# operaciones como iterar día por día o unir estos datos con otras series de tiempo
# que también estén organizadas por fecha.

EST-03: [18.5 19.2 20.  21.5 19.9]
Día 4: [26.5 20.1 21.5 24. ]
Submatriz (2 estaciones, 3 días):
 [[22.3 19.8 24.1]
 [24.  23.5 27.2]]
Shape original: (4, 5)
Shape transpuesta: (5, 4)


---

## Ejercicio 5 — Máscaras booleanas, verificación y fancy indexing (15 pts)

1. Usa una máscara booleana sobre `matriz_temp` para identificar qué lecturas superan los 25°C.
2. Usando `any()`, verifica si **alguna** estación tuvo, en algún día, una lectura mayor a 25°C.
3. Usando `all()`, verifica si la estación `EST-04` (fila correspondiente) tuvo **todos** sus días por encima de los 24°C.
4. Usando fancy indexing, reordena las filas de `matriz_temp` según la lista de prioridad `[2, 0, 3, 1]` (es decir, primero EST-03, luego EST-01, EST-04 y EST-02).

*Por qué importa: filtrar por condición, verificar rápidamente supuestos sobre los datos, y reordenar según un criterio externo son operaciones cotidianas al priorizar qué datos revisar primero.*

In [18]:
# Tu código aquí
# 1. Máscara booleana: qué lecturas superan los 25°C
# matriz_temp > 25 compara CADA elemento de la matriz contra 25, elemento por elemento
# (esto se llama una operación "vectorizada": no hace falta un bucle for).
# El resultado es una nueva matriz de la MISMA forma (4x5), pero llena de valores
# booleanos: True donde la temperatura supera 25, False donde no.
mascara_25 = matriz_temp > 25
print("Máscara (>25°C):\n", mascara_25)

# También podemos usar la máscara para EXTRAER solo los valores que cumplen la condición.
# Al indexar matriz_temp con una máscara booleana, NumPy devuelve un arreglo 1D
# con solo los elementos donde la máscara es True (aplana el resultado).
valores_mayores_25 = matriz_temp[mascara_25]
print("Valores que superan 25°C:", valores_mayores_25)

# 2. ¿Alguna estación tuvo, en algún día, una lectura mayor a 25°C?
# .any() es un método que revisa un arreglo booleano y devuelve True si AL MENOS
# UN elemento es True (equivale a un "O lógico" aplicado a todo el arreglo).
# Aquí lo aplicamos sobre la máscara completa (las 20 lecturas), sin especificar eje,
# por lo que revisa TODA la matriz de una vez.
hay_alguna_mayor_25 = mascara_25.any()
print("¿Alguna lectura > 25°C en toda la red?:", hay_alguna_mayor_25)

# 3. ¿EST-04 tuvo TODOS sus días por encima de 24°C?
# Primero seleccionamos la fila de EST-04. Sabemos que EST-04 es la fila de índice 3
# (fila 0=EST-01, 1=EST-02, 2=EST-03, 3=EST-04).
fila_est04 = matriz_temp[3]

# fila_est04 > 24 genera una máscara booleana 1D (5 elementos, uno por día),
# True donde la temperatura de ese día supera 24°C.
mascara_est04 = fila_est04 > 24

# .all() es un método que revisa un arreglo booleano y devuelve True SOLO SI
# TODOS los elementos son True (equivale a un "Y lógico" aplicado a todo el arreglo).
# Aquí verificamos si TODOS los 5 días de EST-04 superaron los 24°C.
todos_est04_mayor_24 = mascara_est04.all()
print("¿EST-04 tuvo todos sus días > 24°C?:", todos_est04_mayor_24)

# 4. Fancy indexing: reordenar filas según la lista de prioridad [2, 0, 3, 1]
# "Fancy indexing" significa indexar un arreglo usando OTRO arreglo (o lista)
# de índices, en vez de un solo número o un slice.
# Al pasar la lista [2, 0, 3, 1] como índice de filas, NumPy construye una
# NUEVA matriz tomando, en ese orden exacto: la fila 2 (EST-03), luego la fila 0
# (EST-01), luego la fila 3 (EST-04), y finalmente la fila 1 (EST-02).
orden_prioridad = [2, 0, 3, 1]
matriz_reordenada = matriz_temp[orden_prioridad]
print("Matriz reordenada (EST-03, EST-01, EST-04, EST-02):\n", matriz_reordenada)

Máscara (>25°C):
 [[False False False  True False]
 [False False  True False False]
 [False False False False False]
 [ True  True  True False  True]]
Valores que superan 25°C: [26.5 27.2 27.8 26.3 25.9 28.1]
¿Alguna lectura > 25°C en toda la red?: True
¿EST-04 tuvo todos sus días > 24°C?: False
Matriz reordenada (EST-03, EST-01, EST-04, EST-02):
 [[18.5 19.2 20.  21.5 19.9]
 [22.3 19.8 24.1 26.5 21. ]
 [27.8 26.3 25.9 24.  28.1]
 [24.  23.5 27.2 20.1 22.8]]


---

## Ejercicio 6 — Broadcasting y `np.where()` (15 pts)

Cada estación tiene un descalibre de sensor conocido, dado por este arreglo (en el mismo orden que las filas de `matriz_temp`: EST-01, EST-02, EST-03, EST-04):

```python
descalibre = np.array([-0.5, 1.2, 0.0, -1.0])
```

1. Usa broadcasting para aplicar la corrección correspondiente a cada fila de `matriz_temp` (cada estación se ajusta con su propio valor de `descalibre`). Guarda el resultado en una variable nueva, por ejemplo `matriz_corregida`. **Pista:** si intentas sumar `matriz_temp + descalibre` directamente obtendrás un error de formas incompatibles — el broadcasting solo funciona si `descalibre` tiene la forma `(4, 1)` en lugar de `(4,)`, para que cada valor se alinee con una fila completa. Usa `reshape()` (ya lo conoces del Bloque 2) para darle esa forma antes de sumar.
2. Sobre `matriz_corregida`, usa `np.where()` para generar una matriz paralela de etiquetas de texto: `'alta'` si la lectura corregida supera los 25°C, `'normal'` en caso contrario.

In [19]:
descalibre = np.array([-0.5, 1.2, 0.0, -1.0])

# Tu código aquí

# 1. Corrección con broadcasting
# descalibre tiene forma (4,) — un arreglo 1D "plano" con 4 valores, uno por estación.
# matriz_temp tiene forma (4, 5). Si sumamos directamente matriz_temp + descalibre,
# NumPy intentaría alinear descalibre con las COLUMNAS (porque al hacer broadcasting,
# las formas se comparan de derecha a izquierda), pero matriz_temp tiene 5 columnas
# y descalibre tiene 4 elementos -> formas incompatibles -> error.
#
# Para que cada valor de descalibre se aplique a una FILA completa (no a columnas),
# necesitamos que su forma sea (4, 1) en lugar de (4,): una columna con 4 filas.
# reshape(4, 1) reorganiza el arreglo plano en una matriz de 4 filas x 1 columna.
descalibre_columna = descalibre.reshape(4, 1)
print("Forma original de descalibre:", descalibre.shape)
print("Forma tras reshape:", descalibre_columna.shape)

# Ahora sumamos matriz_temp (4,5) + descalibre_columna (4,1).
# Broadcasting "estira" descalibre_columna virtualmente para que tenga forma (4,5),
# repitiendo cada valor a lo largo de toda su fila correspondiente.
# Así, cada estación (fila) se corrige con SU PROPIO valor de descalibre,
# aplicado a las 5 columnas (días) de esa fila.
matriz_corregida = matriz_temp + descalibre_columna
print("Matriz corregida:\n", matriz_corregida)

# 2. Etiquetas con np.where()
# np.where(condicion, valor_si_true, valor_si_false) evalúa la condición
# elemento por elemento sobre toda la matriz, y construye una NUEVA matriz
# de la misma forma, colocando 'alta' donde la condición es True
# y 'normal' donde es False.
etiquetas = np.where(matriz_corregida > 25, "alta", "normal")
print("Etiquetas:\n", etiquetas)

Forma original de descalibre: (4,)
Forma tras reshape: (4, 1)
Matriz corregida:
 [[21.8 19.3 23.6 26.  20.5]
 [25.2 24.7 28.4 21.3 24. ]
 [18.5 19.2 20.  21.5 19.9]
 [26.8 25.3 24.9 23.  27.1]]
Etiquetas:
 [['normal' 'normal' 'normal' 'alta' 'normal']
 ['alta' 'normal' 'alta' 'normal' 'normal']
 ['normal' 'normal' 'normal' 'normal' 'normal']
 ['alta' 'alta' 'normal' 'normal' 'alta']]


---

## Ejercicio 7 — Agregaciones por eje (15 pts)

Usando `matriz_corregida` del Ejercicio 6:

1. Calcula el promedio y la desviación estándar de temperatura **por estación** (usa el `axis` correspondiente).
2. Calcula el promedio de temperatura **por día** (usa el `axis` correspondiente).
3. Sin usar funciones no vistas en el curso (por ejemplo, sin `argmax`), determina **qué día** tuvo el promedio más alto entre estaciones. Sugerencia: obtén el valor máximo de los promedios por día con `.max()`, y compáralo con el arreglo de promedios por día usando `==` para generar una máscara booleana — la posición con `True` te dice qué día fue.

In [20]:
# Tu código aquí
# 1. Promedio y desviación estándar POR ESTACIÓN
# matriz_corregida tiene forma (4, 5): 4 filas (estaciones) x 5 columnas (días).
# axis=1 significa "colapsa/reduce a lo largo de las COLUMNAS", es decir,
# para cada fila (cada estación), calcula un solo valor combinando sus 5 días.
# El resultado es un arreglo 1D de 4 elementos, uno por estación.
promedio_por_estacion = matriz_corregida.mean(axis=1)
std_por_estacion = matriz_corregida.std(axis=1)

print("Promedio por estación:", promedio_por_estacion)
print("Desviación estándar por estación:", std_por_estacion)

# 2. Promedio de temperatura POR DÍA
# axis=0 significa "colapsa/reduce a lo largo de las FILAS", es decir,
# para cada columna (cada día), calcula un solo valor combinando las 4 estaciones.
# El resultado es un arreglo 1D de 5 elementos, uno por día.
promedio_por_dia = matriz_corregida.mean(axis=0)
print("Promedio por día:", promedio_por_dia)

# 3. Determinar qué día tuvo el promedio más alto, sin usar argmax
# .max() sobre promedio_por_dia devuelve el valor máximo entre los 5 promedios diarios
# (un solo número, el más alto de todos).
max_promedio_dia = promedio_por_dia.max()

# promedio_por_dia == max_promedio_dia compara CADA elemento del arreglo contra
# ese valor máximo. Devuelve una máscara booleana de 5 posiciones, con True
# únicamente en la posición (o posiciones) donde el promedio coincide con el máximo.
mascara_dia_max = promedio_por_dia == max_promedio_dia
print("Máscara del día con promedio más alto:", mascara_dia_max)

# np.where(mascara) devuelve una tupla con los ÍNDICES donde la máscara es True.
# Como promedio_por_dia es 1D, np.where(...) devuelve una tupla de un solo arreglo:
# (array([indice]),). Tomamos el índice [0][0] para obtener el número entero.
indice_dia_max = np.where(mascara_dia_max)[0][0]

# Los días están numerados Dia1, Dia2, ..., Dia5, pero los índices empiezan en 0.
# Por eso

Promedio por estación: [22.24 24.72 19.82 25.42]
Desviación estándar por estación: [2.36016949 2.27982455 0.99879928 1.4743134 ]
Promedio por día: [23.075 22.125 24.225 22.95  22.875]
Máscara del día con promedio más alto: [False False  True False False]


---

## Ejercicio 8 — Síntesis: resumen por estación (15 pts)

Usando `map()` y/o comprensión de listas, genera una lista de cadenas de resumen, una por estación, que combine:

- El identificador de estación extraído en el Ejercicio 1.
- El promedio de temperatura corregida de esa estación (Ejercicio 7), formateado a un decimal.
- Su clasificación predominante (`'alta'` o `'normal'`) — usa `np.unique()` sobre las etiquetas de esa estación (del Ejercicio 6) para encontrar cuál aparece con mayor frecuencia entre las dos.

Formato esperado por cadena: `"EST-01: 23.4°C promedio — clasificación predominante: normal"`

Finalmente, ordena la lista de resúmenes de mayor a menor promedio de temperatura usando `sorted()` con `key` y `lambda` (como en la Sesión 4/5), e imprime el resultado ordenado.

In [21]:
# Tu código aquí
# Primero necesitamos los identificadores ÚNICOS de estación, en orden,
# a partir de lo extraído en el Ejercicio 1 (extraidos tiene 20 tuplas,
# 5 repeticiones por estación, así que hay que quedarnos solo con las 4 únicas).
# Comprensión de listas: tomamos el primer elemento (t[0]) de cada tupla en extraidos.
todas_estaciones = [t[0] for t in extraidos]

# dict.fromkeys() elimina duplicados PRESERVANDO el orden de aparición
# (a diferencia de convertir directamente a set(), que no garantiza orden).
# Como cada estación aparece 5 veces seguidas, esto nos deja con
# ['EST-01', 'EST-02', 'EST-03', 'EST-04'].
estaciones_unicas = list(dict.fromkeys(todas_estaciones))
print("Estaciones únicas:", estaciones_unicas)

# Función que, dado un índice de fila (0 a 3), construye la cadena de resumen
# para esa estación. La definimos aparte para poder usarla luego con map().
def resumen_estacion(i):
    # Identificador de estación (ej. 'EST-01'), tomado por posición de la lista única.
    nombre = estaciones_unicas[i]

    # Promedio de temperatura corregida de esa estación, ya calculado en el Ejercicio 7.
    # promedio_por_estacion es un arreglo 1D de 4 valores (uno por fila/estación).
    promedio = promedio_por_estacion[i]

    # etiquetas[i] es la fila i de la matriz de etiquetas del Ejercicio 6:
    # un arreglo de 5 strings ('alta' o 'normal'), uno por día, para esta estación.
    fila_etiquetas = etiquetas[i]

    # np.unique(arreglo, return_counts=True) devuelve dos arreglos:
    #   - los valores únicos encontrados (por ejemplo array(['alta', 'normal']))
    #   - cuántas veces aparece cada uno (por ejemplo array([2, 3]))
    # Es importante pasar return_counts=True; sin ese argumento, np.unique()
    # solo devolvería los valores únicos, sin las frecuencias.
    valores, conteos = np.unique(fila_etiquetas, return_counts=True)

    # Para encontrar cuál valor es el más frecuente, comparamos conteos contra
    # su propio máximo (mismo truco booleano del Ejercicio 7), y usamos esa
    # máscara para indexar 'valores' y obtener el string correspondiente.
    clasificacion_predominante = valores[conteos == conteos.max()][0]

    # f-string: construye la cadena final con el formato pedido.
    # ":.1f" formatea el número con exactamente 1 decimal.
    return f"{nombre}: {promedio:.1f}°C promedio — clasificación predominante: {clasificacion_predominante}"

# map() aplica resumen_estacion() a cada índice de la lista [0, 1, 2, 3]
# (uno por cada una de las 4 estaciones), generando las 4 cadenas de resumen.
# list() convierte el resultado (un objeto map perezoso) en una lista concreta.
resumenes = list(map(resumen_estacion, range(len(estaciones_unicas))))

print("Resúmenes sin ordenar:")
for r in resumenes:
    print(r)

# Ordenar de MAYOR a MENOR promedio de temperatura.
# sorted() recibe la lista a ordenar y una función 'key' que le dice,
# para cada elemento, qué valor usar como criterio de comparación.
# lambda r: float(r.split(":")[1].split("°C")[0]) extrae el número de la cadena:
#   - r.split(":")[1] toma la parte después del primer ":", ej. " 23.4°C promedio — ..."
#   - .split("°C")[0] corta justo antes de "°C", dejando " 23.4"
#   - float(...) lo convierte a número para poder compararlo/ordenarlo.
# reverse=True ordena de mayor a menor (en vez del orden ascendente por defecto).
resumenes_ordenados = sorted(resumenes, key=lambda r: float(r.split(":")[1].split("°C")[0]), reverse=True)

print("\nResúmenes ordenados (mayor a menor promedio):")
for r in resumenes_ordenados:
    print(r)

Estaciones únicas: ['EST-01', 'EST-02', 'EST-03', 'EST-04']
Resúmenes sin ordenar:
EST-01: 22.2°C promedio — clasificación predominante: normal
EST-02: 24.7°C promedio — clasificación predominante: normal
EST-03: 19.8°C promedio — clasificación predominante: normal
EST-04: 25.4°C promedio — clasificación predominante: alta

Resúmenes ordenados (mayor a menor promedio):
EST-04: 25.4°C promedio — clasificación predominante: alta
EST-02: 24.7°C promedio — clasificación predominante: normal
EST-01: 22.2°C promedio — clasificación predominante: normal
EST-03: 19.8°C promedio — clasificación predominante: normal


---

## Antes de entregar

- Verifica que el notebook completo corra de principio a fin sin errores (**Entorno de ejecución → Ejecutar todas**).
- Confirma que no usaste pandas ni scikit-learn.
- Comparte el notebook con permiso de **edición** para el docente.
- Revisa que tu nombre y número de control estén en la primera celda.